# GWjax — Full Parameter Estimation on Colab GPU

This notebook walks through a complete BBH parameter-estimation run with **GWjax** end-to-end:

1. Install `gwjax` (with the `[data]` extra for real-data ingest).
2. Verify a CUDA device is visible to JAX.
3. Build a 2-detector H1/L1 network and either:
    - inject a synthetic IMRPhenomD signal, **or**
    - fetch real GW150914 strain from GWOSC.
4. Sample the posterior with the BlackJAX-NS nested sampler.
5. Plot the joint posterior with `corner`.

**Runtime → Change runtime type → T4 GPU** before running. Total wall-clock is a few minutes on a T4.

## 1. Install GWjax

In [ ]:
# Colab GPU images already ship JAX with CUDA. The `[data]` extra adds gwpy + gwosc.
!pip install -q "gwjax[data] @ git+https://github.com/Saulosoares/GWjax.git"
!pip install -q corner

In [ ]:
import jax, jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

# Enable double precision — important for likelihood accuracy.
jax.config.update("jax_enable_x64", True)

import gwjax
print("gwjax version :", gwjax.__version__)
print("JAX devices   :", jax.devices())

## 2. Build a detector network

We use a 4-second segment at 2048 Hz, analysing the 20–512 Hz band. The `TimeFrequencyGrid` is shared across all detectors in the network.

In [ ]:
DURATION       = 4.0
SAMPLING_RATE  = 2048.0
F_MIN, F_MAX   = 20.0, 512.0

grid = gwjax.TimeFrequencyGrid(
    duration=DURATION, sampling_rate=SAMPLING_RATE,
    f_min=F_MIN, f_max=F_MAX,
)
network = gwjax.Network.from_names(["H1", "L1"], grid)
print(grid)
print(network)

## 3a. Synthetic injection (default — fast, deterministic)

Generate coloured Gaussian noise at the aLIGO design PSD, then inject a GW150914-like IMRPhenomD signal. The network optimal SNR ends up around 19.

In [ ]:
TRUE_PARAMS = dict(
    m1=35.0, m2=30.0, chi_1=0.0, chi_2=0.0,
    distance=410.0, inclination=0.4,
    tc=0.0, phi_c=0.0,
    ra=1.375, dec=-1.21, psi=0.0,
)

network.generate_noise(seed=0)
waveform_fn = gwjax.build_ripplegw_waveform_fn("IMRPhenomD", f_ref=20.0)

hp, hc = waveform_fn(TRUE_PARAMS, grid.frequency_domain_array)
h_dict = network.project_waveform(
    hp, hc, TRUE_PARAMS["ra"], TRUE_PARAMS["dec"], TRUE_PARAMS["psi"], gmst=0.0,
)
for name, snr in network.optimal_snr(h_dict).items():
    print(f"  injected {name} SNR = {float(snr):.1f}")
print(f"  network SNR = {float(network.network_optimal_snr(h_dict)):.1f}")

network.inject_signal(h_dict, domain="fd")

## 3b. (Optional) Real GW150914 strain from GWOSC

Uncomment and run this cell **instead of 3a** to PE the real event. The `attach_event_to_network` helper:

- downloads the H1 and L1 strain around the GW150914 trigger (`gwpy.TimeSeries.fetch_open_data`),
- estimates each IFO's PSD from an adjacent off-source segment via Welch,
- crops + resamples the on-source segment, and attaches everything to the network.

If you ran cell 3a, **reset the network** first by re-running cell 2.

In [ ]:
# Uncomment to use real data instead of the synthetic injection:
#
# # GW150914 standard analysis: 4 s @ 4 kHz, 20–1024 Hz.
# grid = gwjax.TimeFrequencyGrid(
#     duration=4.0, sampling_rate=4096.0, f_min=20.0, f_max=1024.0,
# )
# network = gwjax.Network.from_names(["H1", "L1"], grid)
# gwjax.compat.attach_event_to_network(
#     network, "GW150914",
#     estimate_psd=True, psd_segment_duration=32.0, psd_offset=8.0,
# )
# TRUE_PARAMS = None   # we don't know the truth; the published best-fit can be used as plot reference.

## 4. Set up the nested sampler

Eight-dimensional uniform prior over component masses, distance, inclination, and the four sky/time parameters. Spins and the coalescence phase are fixed (set them in `param_bounds` instead of `fixed_params` to sample them too).

In [ ]:
PARAM_BOUNDS = {
    "m1":          (10.0, 80.0),
    "m2":          (10.0, 80.0),
    "distance":    (50.0, 1500.0),
    "inclination": (0.0, float(jnp.pi)),
    "ra":          (0.0, 2.0 * float(jnp.pi)),
    "dec":         (-float(jnp.pi) / 2, float(jnp.pi) / 2),
    "psi":         (0.0, float(jnp.pi)),
    "tc":          (-0.05, 0.05),
}
FIXED_PARAMS = {"chi_1": 0.0, "chi_2": 0.0, "phi_c": 0.0}

sampler = gwjax.GWjaxNestedSampler(
    network       = network,
    waveform_fn   = waveform_fn,
    param_bounds  = PARAM_BOUNDS,
    fixed_params  = FIXED_PARAMS,
    gmst          = 0.0,
)
print(f"sampling dimension: {len(sampler.param_bounds)}")

## 5. Run the sampler

On a T4 GPU a 400 live-point, 3 k-iteration run takes a few minutes. Reduce `num_live` and `max_iterations` for a quick smoke test (with looser posteriors).

In [ ]:
import time

t0 = time.perf_counter()
result = sampler.run(
    rng_key               = jax.random.PRNGKey(0),
    num_live              = 400,
    num_inner_steps       = 25,
    max_iterations        = 3000,
    log_dlogz_target      = -3.0,
    num_posterior_samples = 2000,
    verbose               = True,
)
elapsed = time.perf_counter() - t0

print(f"\nNS finished in {elapsed:.1f} s, {result.n_iterations} iterations")
print(f"  log Z = {result.logZ:+.3f} ± {result.logZ_err:.3f}")
print(f"  ESS   = {result.ess:.1f}")

## 6. Posterior summary

In [ ]:
print(f"  {'param':12s}  {'median':>10s}  {'-1σ':>8s}  {'+1σ':>8s}  truth")
for name in sampler.param_bounds:
    s = np.asarray(result.posterior_samples[name])
    lo, mid, hi = np.percentile(s, [16, 50, 84])
    if TRUE_PARAMS is not None:
        truth_str = f"{TRUE_PARAMS[name]:+.3f}"
    else:
        truth_str = "—"
    print(f"  {name:12s}  {mid:+10.3f}  {mid-lo:8.3f}  {hi-mid:8.3f}  {truth_str}")

## 7. Corner plot

In [ ]:
import corner

names  = list(sampler.param_bounds.keys())
data   = np.column_stack([np.asarray(result.posterior_samples[n]) for n in names])
truths = [TRUE_PARAMS[n] for n in names] if TRUE_PARAMS is not None else None

fig = corner.corner(
    data, labels=names, truths=truths,
    quantiles=[0.16, 0.5, 0.84], show_titles=True,
    title_kwargs={"fontsize": 10},
)
fig.set_size_inches(11, 11)
plt.show()

## What next?

- **Use your own waveform**: wrap any pure-JAX `hp, hc = wf(freqs, **params)` in `gwjax.CustomWaveform` and pass it as `waveform_fn=` (no need for ripplegw).
- **Add more detectors**: include `"V1"`, `"K1"`, or even `"ET"`/`"CE"` in the `Network.from_names` list.
- **Sample spins**: move `chi_1`, `chi_2` from `fixed_params` to `param_bounds` (e.g. each in `(-0.9, 0.9)`).
- **Custom PSDs**: pass a path to an ASCII PSD file via `psd_model=` when constructing an `Interferometer`, or estimate from data with `gwjax.psd_from_data`.
- **Run locally**: the same workflow ships as `examples/run_pe_local.py` — see the README for CLI options.